# Visitor Geolocation Tracker: FastAPI + SQLite Data Pipeline

| | |
|---|---|
| **Type** | Full-stack data-capture project (backend + database) |
| **Domain** | Data engineering · Web APIs · Geospatial telemetry |
| **Tools** | Python, FastAPI, SQLAlchemy ORM, SQLite, Jinja2, Leaflet.js, browser Geolocation API, ip-api.com, `user-agents`, Gunicorn |
| **Deployment** | Render (Procfile-based web service) |
| **Date** | 2026 |
| **Repository** | [GitHub repo link] |
| **Live demo** | [add link once the consent notice below is in place] |

> **TL;DR:** A FastAPI service that records, for each visit, the **client IP, IP-derived location, browser GPS
> position with accuracy, and device/OS/browser details** into a SQLite database through SQLAlchemy. A
> Basic-Auth-protected admin area lets the owner view records, export them to CSV, and wipe the table. The project
> is a compact end-to-end example of **collecting → enriching → storing → exporting** data, and of the privacy
> questions that come with it.

::::{grid} 2 2 4 4
:gutter: 2

:::{grid-item-card} 1 table · 16 fields
:text-align: center
relational schema managed by an ORM
:::

:::{grid-item-card} 3
:text-align: center
data sources merged per record: browser GPS, IP-lookup API, User-Agent
:::

:::{grid-item-card} 7
:text-align: center
HTTP routes (4 public, 3 admin-protected)
:::

:::{grid-item-card} CSV
:text-align: center
streamed export for downstream analysis
:::

::::

## 1. Problem Statement

Web apps generate valuable telemetry (where visitors are, what device they use, how precise the location is),
but that data has to be **captured reliably, enriched, stored in a queryable form, and exported**.

**Objectives**

- Capture precise coordinates via the browser Geolocation API *and* a fallback coarse location from the IP address.
- Enrich each event with ISP/city/region/country and a parsed device profile.
- Persist events in a relational database with a defined schema.
- Provide a protected interface to inspect, export, and delete records.
- Deploy the service so it can be tested from real devices.

## 2. Data & Storage

### Data flow

```text
Browser ── permission prompt ──▶ JavaScript Geolocation (lat, lon, accuracy)
   │                                        │
   │                              POST /log-location  (JSON)
   ▼                                        ▼
FastAPI  ── client IP ──▶ ip-api.com lookup (city, region, country, ISP)
         ── User-Agent header ──▶ `user-agents` parser (device, OS, browser)
         ── server clock ──▶ UTC timestamp
                                            │  SQLAlchemy ORM (session per request)
                                            ▼
                                  SQLite  ·  table `visitor_logs`
                                            │
        Admin (HTTP Basic) ──▶ /logs (HTML table) · /download-logs (CSV) · /delete-logs
```

### Storage design

| Aspect | Choice |
|---|---|
| **Database** | SQLite file (`logs.db`): embedded, zero-config, ideal for prototypes |
| **Access layer** | SQLAlchemy declarative ORM; `create_engine("sqlite:///logs.db", connect_args={"check_same_thread": False})` so FastAPI worker threads can share the file |
| **Schema creation** | `Base.metadata.create_all()` on startup creates the table if missing |
| **Sessions** | One `SessionLocal()` per request, committed and closed explicitly |
| **Keys & indexes** | Integer primary key `id` with an index |
| **Time handling** | Timestamps recorded in **UTC** |
| **Export** | Rows written with `csv.writer` into an in-memory buffer and returned as a `StreamingResponse` (`Content-Disposition: attachment`) |
| **Ad-hoc inspection** | Helper script runs raw SQL (e.g. latest 5 rows by timestamp) through `sqlite3` |
| **Repo hygiene** | The committed database is essentially empty (a single local test row) |

### Schema (`visitor_logs`)

In [1]:
import sqlite3
import pandas as pd

# DDL copied from the project's logs.db (sqlite_master)
DDL = """
CREATE TABLE visitor_logs (
    id INTEGER NOT NULL,
    ip_address VARCHAR(50),
    city TEXT, region TEXT, country TEXT, isp TEXT,
    latitude FLOAT, longitude FLOAT,
    user_agent TEXT,
    device_brand VARCHAR(50), os VARCHAR(50), browser TEXT,
    timestamp DATETIME,
    bssid VARCHAR(50),
    location_source VARCHAR(20),
    accuracy FLOAT,
    PRIMARY KEY (id)
);
CREATE INDEX ix_visitor_logs_id ON visitor_logs (id);
"""
source = {
    "id": "auto-generated", "ip_address": "server (request.client.host)",
    "city": "ip-api.com", "region": "ip-api.com", "country": "ip-api.com", "isp": "ip-api.com",
    "latitude": "browser GPS", "longitude": "browser GPS", "user_agent": "request header",
    "device_brand": "parsed from User-Agent", "os": "parsed from User-Agent", "browser": "parsed from User-Agent",
    "timestamp": "server clock (UTC)", "bssid": "client-reported (optional)",
    "location_source": "client-reported (GPS / ip-only)", "accuracy": "browser GPS (metres)",
}
con = sqlite3.connect(":memory:")
con.executescript(DDL)
schema = pd.read_sql("PRAGMA table_info(visitor_logs)", con)[["name", "type", "pk"]]
schema["source"] = schema["name"].map(source)
schema.columns = ["column", "type", "primary key", "filled from"]
con.close()
schema

,column,type,primary key,filled from
0,id,INTEGER,1,auto-generated
1,ip_address,VARCHAR(50),0,server (request.client.host)
2,city,TEXT,0,ip-api.com
3,region,TEXT,0,ip-api.com
4,country,TEXT,0,ip-api.com
5,isp,TEXT,0,ip-api.com
6,latitude,FLOAT,0,browser GPS
7,longitude,FLOAT,0,browser GPS
8,user_agent,TEXT,0,request header
9,device_brand,VARCHAR(50),0,parsed from User-Agent


## 3. Data Engineering & Preprocessing

| Step | Implementation | Purpose |
|---|---|---|
| **Client-side capture** | `getCurrentPosition` with `enableHighAccuracy` and a 10-second timeout; result POSTed as JSON | Get GPS-grade coordinates when the user allows it |
| **Graceful fallback** | If GPS is denied or times out, the page posts `location_source = "ip-only"` and coordinates stay `NULL` | Keeps the record while preserving honest missing values |
| **Data enrichment** | Server calls `ip-api.com/json/<ip>` and stores city, region, country, ISP; missing fields default to `"Unknown"` | Adds coarse geography even without GPS |
| **Device profiling** | `user_agents.parse()` extracts device family, OS family, and browser family from the raw header | Turns an unstructured string into analysable categories |
| **Type handling** | Coordinates and accuracy stored as `FLOAT`; timestamps recorded in UTC | Enables numeric and time-based analysis |
| **Quality signal** | The `accuracy` radius (metres) is stored with every GPS point | Lets analysts filter out low-precision locations |
| **Presentation layer** | Leaflet + OpenStreetMap map showing the captured position with an accuracy circle | Visual validation of what was captured |
| **Export shaping** | Explicit header row and column order in the CSV writer | Clean hand-off to pandas, R, or Excel |

**Routes**

| Method | Route | Access | Purpose |
|---|---|---|---|
| GET | `/` | public | Landing page requesting location permission |
| GET | `/tracking` | public | Map page that triggers capture |
| GET | `/get-ip-info` | public | JSON of IP-based geolocation |
| POST | `/log-location` | public | Receives GPS payload, enriches, inserts row |
| GET | `/logs` | admin (Basic Auth) | HTML table, newest first |
| GET | `/download-logs` | admin (Basic Auth) | CSV export |
| POST | `/delete-logs` | admin (Basic Auth) | Delete all rows |

## 4. Outcome

- A working **capture → enrich → store → export** pipeline, deployed as a web service on Render.
- A normalised, typed schema that can go straight into pandas (`pd.read_sql`) for analysis: visits per hour, device/OS mix, location-accuracy distribution.
- Practical exposure to **ORM-based persistence, HTTP authentication, third-party API enrichment, and CSV streaming**.

Because the repository intentionally ships without visitor data, there are no usage statistics to report here.

## 5. Privacy & Ethics Considerations

Precise location and IP addresses are **personal data** (for example, under the Philippine Data Privacy Act of
2012, RA 10173). Responsible use requires more than technical capture:

- **Consent first.** The browser's permission prompt covers GPS only. IP address and device details are still
  logged in the fallback path, so a clear on-page notice and an explicit opt-in should come *before* any logging.
- **Data minimisation.** Store only the fields the analysis needs; consider coarsening coordinates.
- **Retention and deletion.** Define how long records are kept; the admin delete route is a start, but automatic expiry is better.
- **Access control.** Restrict the admin area, keep credentials in environment variables, and rotate them.
- **Purpose limitation.** Use the tracker only with participants who know what is collected and why.

## 6. Data Science Concepts Demonstrated

| Concept | Where it appears |
|---|---|
| **Relational data modelling** | Typed columns, primary key, index, and an explicit schema |
| **ORM & persistence** | SQLAlchemy sessions, declarative models, automatic table creation |
| **Data collection pipeline** | Client sensor → API → enrichment → storage |
| **Data enrichment** | IP geolocation and User-Agent parsing |
| **Missing-data handling** | `NULL` coordinates when GPS is unavailable; `"Unknown"` defaults |
| **Data quality** | Storing the accuracy radius alongside each coordinate |
| **REST API design** | Public collection endpoint and protected admin endpoints |
| **Data export & interoperability** | Streaming CSV download |
| **Deployment** | Gunicorn/Procfile web service on Render |
| **Privacy & ethics** | Consent, minimisation, retention, access control |

## 7. Limitations & Next Steps

- **Consent and notice.** Add a consent banner and log only after opt-in (see Section 5).
- **Accurate client IPs behind a proxy.** `request.client.host` may return the platform's proxy address; read
  `X-Forwarded-For` (with trusted-proxy settings) so the stored IP is the visitor's.
- **Durable storage.** SQLite files on PaaS disks can be ephemeral across deploys. Move to a managed
  **PostgreSQL** instance (only the SQLAlchemy URL changes) and add **Alembic** migrations.
- **Schema growth.** Normalise into `visitors`, `sessions`, and `devices`; index `timestamp`; add constraints.
- **Robust enrichment.** Use async HTTP with timeouts, retries, caching, and an HTTPS geolocation provider.
- **Validation and security.** Validate request bodies with Pydantic; store secrets outside the code; use
  hashed credentials with constant-time comparison; add rate limiting.
- **Finish the admin features.** The log page has a search box that is not wired to a server-side filter yet,
  and the CSV export omits `bssid`, `location_source`, and `accuracy`.
- **Analytics layer.** Add pandas/SQL summaries and a small dashboard (visits by hour, device mix, accuracy distribution).
- **Tests.** Add `pytest` with FastAPI's `TestClient` for the routes and the enrichment functions.